# Solar Filament Segmentation 2026 — Full Pipeline
End-to-end walkthrough: preprocessing → dataset → model → loss → training → instancing → Panoptic Quality → RLE submission.

Set `DATA` to your unzipped `MAGFiLO_1.0_Kaggle_2026` folder.

In [ ]:
import os, sys, numpy as np, cv2, torch, matplotlib.pyplot as plt
sys.path.append('..')  # repo root
DATA = 'data'  # -> data/train/... and data/test/...
ANN  = f'{DATA}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'
IMGS = f'{DATA}/train/train_images'
TEST = f'{DATA}/test/test_images'
device = 'cuda' if torch.cuda.is_available() else 'cpu'; device

## 1. Preprocessing
Limb detection → radial limb-darkening flattening → CLAHE → Frangi vesselness, stacked into a 3-channel input. Filaments are dark absorption features, so the vesselness channel runs on the inverted image.

In [ ]:
from src.preprocessing import preprocess_gong_halpha
sample = sorted(os.listdir(IMGS))[0]
tensor, disk = preprocess_gong_halpha(f'{IMGS}/{sample}')
fig,ax = plt.subplots(1,3,figsize=(14,5))
for i,t in enumerate(['flattened','CLAHE','vesselness']):
    ax[i].imshow(tensor[...,i],cmap='gray'); ax[i].set_title(t); ax[i].axis('off')
plt.show()

## 2. Dataset
COCO loader that pools all annotators sharing a `file_name` and rasterises spines. `group_key` (YYYYMMDD) drives the leakage-free split.

In [ ]:
from src.dataset import MagfiloDataset
ds = MagfiloDataset(ANN, IMGS, img_size=512, return_instances=True)
print('images:', len(ds))
item = ds[0]
print('fg px:', int(item['fg'].sum()), 'spine px:', int(item['spine'].sum()),
      'gt instances:', len(item['instances']))

## 3. Model & Loss
U-Net with foreground + spine heads. Loss = BCE + Dice + λ·clDice (centerline connectivity for barbs) plus an auxiliary spine term.

In [ ]:
from src.models.unet import UNetSpine
from src.models.losses import CombinedLoss
model = UNetSpine().to(device)
crit  = CombinedLoss(w_cldice=0.5, w_spine=0.5)
print('params:', sum(p.numel() for p in model.parameters()))

## 4. Train
Use the script for full runs (grouped split, PQ-based checkpointing):
```bash
python scripts/train.py --ann_json {ANN} --images_dir {IMGS}
```

In [ ]:
# quick single-batch sanity step
x = item['image'].unsqueeze(0).to(device)
tgt = {'fg':item['fg'].unsqueeze(0).to(device),'spine':item['spine'].unsqueeze(0).to(device)}
loss,_ = crit(model(x), tgt); loss.backward()
print('loss:', float(loss))

## 5. Instancing + Panoptic Quality
Spine-guided watershed splits the semantic map into instances; PQ = SQ×RQ scores them against the fused ground truth.

In [ ]:
from src.instancing import instances_from_semantic
from src.metrics import panoptic_quality
model.eval()
with torch.no_grad(): out = model(x)
seg = torch.sigmoid(out['seg'])[0,0].cpu().numpy()
sp  = torch.sigmoid(out['spine'])[0,0].cpu().numpy()
preds,scores = instances_from_semantic(seg, sp)
gts = [cv2.resize(g,(512,512),interpolation=cv2.INTER_NEAREST) for g in item['instances']]
print(panoptic_quality(preds, gts))

## 6. Submission
Run inference over the test set with rotational TTA and write the RLE CSV:
```bash
python scripts/predict.py --input_dir {TEST} --output_csv submission/submission.csv
```
Validate the file against the organizers' self-evaluation notebook before uploading.